# DiT Layout Bench: Colab 학습 · MLflow · 검증

이 노트북은 다음 경로를 위에서 아래로 실행합니다.

1. 저장소와 Colab GPU 환경 준비
2. Hugging Face PubLayNet을 COCO 형식으로 일부 변환
3. `scripts/publaynet_subset.py`로 재현 가능한 train/val subset 생성
4. DiT pretrained checkpoint 검사와 smoke-test 설정 생성
5. DINO 또는 Cascade R-CNN 학습
6. MLflow history, loss 전달, train/eval namespace 검증
7. checkpoint resume와 동일 MLflow run 연결 검증
8. MLflow 계약 테스트 실행

> Colab 메뉴에서 **런타임 → 런타임 유형 변경 → GPU**를 먼저 선택하세요. 기본 backend는 DINO입니다. Cascade R-CNN은 현재 Colab의 PyTorch/CUDA ABI에 맞춰 Detectron2를 소스 빌드해야 합니다.

In [ ]:
# 사용자 설정: 이 셀만 먼저 확인하세요.
from pathlib import Path

REPO_URL = "https://github.com/whansk50/dino_dit.git"
BRANCH = "main"
PROJECT_DIR = Path("/content/dino_dit")

BACKEND = "dino"  # "dino" 또는 "cascade_rcnn"
DEVICE_ID = 0
SEED = 42
IMAGE_SIZE = 256
TRAIN_IMAGES = 8
VAL_IMAGES = 4
SOURCE_IMAGES_PER_SPLIT = 16  # subset 원본으로 먼저 받을 이미지 수
BATCH_SIZE = 1  # Colab 단일 GPU의 global batch
FRESH_EPOCHS = 1
RESUME_EPOCHS = 2
RUN_RESUME_TEST = True
RUN_FULL_UNIT_TESTS = False
MOUNT_GOOGLE_DRIVE = False

WORK_ROOT = Path("/content/dit-layout-colab")
SOURCE_DATA_ROOT = WORK_ROOT / f"publaynet-source-{SOURCE_IMAGES_PER_SPLIT}"
SUBSET_ROOT = WORK_ROOT / f"publaynet-subset-t{TRAIN_IMAGES}-v{VAL_IMAGES}-s{SEED}"
OUTPUT_DIR = WORK_ROOT / f"output-{BACKEND}"
WEIGHTS_DIR = OUTPUT_DIR / "weights"
MLFLOW_DB = WORK_ROOT / "mlflow.db"
GENERATED_CONFIG = WORK_ROOT / f"{BACKEND}-colab.yaml"

# 권장: checkpoint를 Google Drive에 두고 이 경로만 바꾸세요.
PRETRAINED_PATH = Path("/content/dit-base-224-p16-500k-62d53a.pth")
# 직접 접근 가능한 배포 URL이 있을 때만 지정합니다. 빈 문자열이면 다운로드하지 않습니다.
PRETRAINED_URL = ""

assert BACKEND in {"dino", "cascade_rcnn"}
assert SOURCE_IMAGES_PER_SPLIT >= max(TRAIN_IMAGES, VAL_IMAGES)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print(f"backend={BACKEND}, work_root={WORK_ROOT}")

## 1. 런타임과 저장소 준비

이 프로젝트는 `torchrun`을 사용하지 않습니다. 단일 GPU에서는 `--devices 0`, 다중 GPU 환경에서는 `--devices 0,1`처럼 전달하면 프로젝트 내부의 PyTorch launcher가 process group과 DDP를 관리합니다.

In [ ]:
import os
import subprocess
import sys
import torch

assert torch.cuda.is_available(), "GPU 런타임을 선택해야 합니다."
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(DEVICE_ID))
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], check=True)

In [ ]:
def run(command, *, cwd=None):
    print("+", " ".join(map(str, command)))
    return subprocess.run([str(value) for value in command], cwd=cwd, check=True)

if not (PROJECT_DIR / ".git").is_dir():
    run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, PROJECT_DIR])
else:
    print("기존 checkout을 재사용합니다:", PROJECT_DIR)

run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{PROJECT_DIR}[test]", "datasets>=2.18,<4", "ninja"])
os.chdir(PROJECT_DIR)
print(run(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR))

In [ ]:
# backend별 native dependency 준비
if BACKEND == "dino":
    run(["bash", "scripts/build_dino_ops.sh"], cwd=PROJECT_DIR)
else:
    # 실행 시점의 HEAD를 먼저 SHA로 고정해 설치 중 source가 바뀌지 않게 합니다.
    detectron_repository = "https://github.com/facebookresearch/detectron2.git"
    detectron_commit_file = WORK_ROOT / "detectron2-commit.txt"
    if detectron_commit_file.is_file():
        resolved = detectron_commit_file.read_text(encoding="utf-8").strip()
    else:
        resolved = subprocess.run(
            ["git", "ls-remote", detectron_repository, "HEAD"],
            check=True, capture_output=True, text=True,
        ).stdout.split()[0]
        detectron_commit_file.write_text(resolved, encoding="utf-8")
    run([sys.executable, "-m", "pip", "install", "-q", f"git+{detectron_repository}@{resolved}"])
    import detectron2
    from detectron2 import _C  # compiled extension까지 즉시 검사
    print("Detectron2 commit:", resolved)

print("backend dependency 준비 완료")

## 2. PubLayNet 다운로드와 subset 생성

먼저 `download_from_huggingface.py`가 Hugging Face row를 COCO 구조로 변환합니다. 그 결과에서 `create_publaynet_subset()`이 annotation이 있는 이미지를 우선해 seed 기반으로 선택하고, 원본 이미지는 수정하지 않은 채 symlink로 subset을 구성합니다.

전체 PubLayNet이 Drive 등에 이미 있다면 다음 다운로드 셀을 건너뛰고 `SOURCE_DATA_ROOT`를 해당 경로로 바꾸면 됩니다.

In [ ]:
download_command = [
    sys.executable,
    "scripts/download_from_huggingface.py",
    "--output-dir", str(SOURCE_DATA_ROOT),
    "--max-images-per-split", str(SOURCE_IMAGES_PER_SPLIT),
    "--progress-every", "4",
    "--resume",
]
run(download_command, cwd=PROJECT_DIR)

In [ ]:
import json
from scripts.publaynet_subset import create_publaynet_subset

subset_manifest = SUBSET_ROOT / "subset-request.json"
expected_subset = {
    "source_root": str(SOURCE_DATA_ROOT.resolve()),
    "train_images": TRAIN_IMAGES, "val_images": VAL_IMAGES, "seed": SEED,
}
if subset_manifest.is_file():
    actual_subset = json.loads(subset_manifest.read_text(encoding="utf-8"))
    if actual_subset != expected_subset:
        raise ValueError(f"기존 subset 설정이 다릅니다: {actual_subset}")
elif SUBSET_ROOT.exists() and any(SUBSET_ROOT.iterdir()):
    raise ValueError(f"manifest 없는 기존 subset입니다. 새 WORK_ROOT를 사용하세요: {SUBSET_ROOT}")
else:
    SUBSET_ROOT.mkdir(parents=True, exist_ok=True)
    subset_manifest.write_text(json.dumps(expected_subset, indent=2), encoding="utf-8")

subset_requests = (("train", TRAIN_IMAGES, SEED), ("val", VAL_IMAGES, SEED + 1))
for split, image_count, split_seed in subset_requests:
    annotation_path = SUBSET_ROOT / "annotations" / f"{split}.json"
    if annotation_path.is_file():
        print(f"[{split}] 기존 subset 재사용: {annotation_path}")
        continue
    create_publaynet_subset(
        SOURCE_DATA_ROOT,
        SUBSET_ROOT,
        split=split,
        image_count=image_count,
        seed=split_seed,
    )
    print(f"[{split}] subset 생성: images={image_count}, seed={split_seed}")

In [ ]:
import json
from dit_layout_bench.data import validate_publaynet

validate_publaynet(SUBSET_ROOT)
for split in ("train", "val"):
    document = json.loads((SUBSET_ROOT / "annotations" / f"{split}.json").read_text())
    links = list((SUBSET_ROOT / split).glob("**/*"))
    print(
        split,
        f"images={len(document['images'])}",
        f"annotations={len(document['annotations'])}",
        f"symlinks={sum(path.is_symlink() for path in links)}",
    )
print("PubLayNet category/layout 계약 검증 완료")

## 3. DiT pretrained checkpoint

새 학습에는 IIT-CDIP로 self-supervised pretraining된 DiT-base/16 checkpoint가 필요합니다. 라이선스와 배포 조건을 확인한 파일을 Drive에 저장하고 `PRETRAINED_PATH`를 지정하는 방식을 권장합니다. 공식 모델 설명은 [Microsoft UniLM DiT README](https://github.com/microsoft/unilm/tree/master/dit)를 참고하세요.

직접 접근 가능한 URL을 보유한 경우에만 첫 설정 셀의 `PRETRAINED_URL`을 채우면 아래 셀이 다운로드합니다.

In [ ]:
from urllib.request import urlretrieve

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if not PRETRAINED_PATH.is_file() and PRETRAINED_URL:
    PRETRAINED_PATH.parent.mkdir(parents=True, exist_ok=True)
    print("checkpoint 다운로드 중...")
    urlretrieve(PRETRAINED_URL, PRETRAINED_PATH)

if not PRETRAINED_PATH.is_file():
    raise FileNotFoundError(
        f"DiT checkpoint가 없습니다: {PRETRAINED_PATH}\n"
        "Google Drive 경로를 PRETRAINED_PATH에 지정하거나 PRETRAINED_URL을 설정하세요."
    )
print(f"checkpoint={PRETRAINED_PATH} ({PRETRAINED_PATH.stat().st_size / 2**20:.1f} MiB)")

## 4. Colab smoke-test 설정 생성

프로젝트의 실제 YAML을 불러온 뒤 경로와 실행량만 줄입니다. DINO는 decoder/encoder를 한 층으로 축소하지만 동일한 backbone, loss, evaluation, checkpoint, MLflow 경로를 사용합니다.

In [ ]:
import yaml
from dit_layout_bench.config import load_settings

base_config = PROJECT_DIR / "configs" / f"{BACKEND}_train.yaml"
settings = load_settings(base_config, detector=BACKEND)
settings["run"].update(device="cuda", seed=SEED, num_workers=0, amp=True)
settings["paths"].update(
    data_root=str(SUBSET_ROOT),
    output_dir=str(OUTPUT_DIR),
    weights_dir=str(WEIGHTS_DIR),
    pretrained=str(PRETRAINED_PATH.resolve()),
)
settings["training"].update(
    batch_size=BATCH_SIZE,
    epochs=FRESH_EPOCHS,
    warmup_iters=0,
    evaluate_every_epochs=1,
)
settings["input"].update(short_edge_scales=[IMAGE_SIZE], max_long_edge=IMAGE_SIZE)
settings["dit"].update(drop_path=0.0, use_checkpoint=True)
settings["tracking"].update(
    enabled=True,
    tracking_uri=f"sqlite:///{MLFLOW_DB}",
    experiment_name=f"colab-{BACKEND}-smoke",
    run_name=f"{BACKEND}-fresh-resume",
    log_every_steps=1,
)

if BACKEND == "dino":
    settings["training"].update(persistent_workers=False)
    settings["dino"].update(
        enc_layers=1, dec_layers=1, dim_feedforward=256,
        num_queries=10, num_select=10, dn_number=2,
        fused_optimizer=True, ddp_static_graph=False,
    )
else:
    settings["cascade_rcnn"].update(
        roi_batch_size_per_image=32, rpn_batch_size_per_image=32
    )

GENERATED_CONFIG.parent.mkdir(parents=True, exist_ok=True)
GENERATED_CONFIG.write_text(
    yaml.safe_dump(settings, sort_keys=False, allow_unicode=True), encoding="utf-8"
)
print(GENERATED_CONFIG.read_text())

## 5. Fresh training

학습 과정에서 rank 0이 MLflow run을 만들고, `effective-config.yaml`, runtime batch 정보, iteration loss/LR, epoch 평균, COCO AP를 기록합니다. validation 성공 후 `weights/recent.pth`가 resume target으로 생성됩니다.

In [ ]:
checkpoint = WEIGHTS_DIR / "recent.pth"
if checkpoint.exists():
    raise FileExistsError(
        f"fresh training checkpoint가 이미 있습니다: {checkpoint}. "
        "새 실험은 첫 설정 셀의 WORK_ROOT를 변경해서 실행하세요."
    )

train_command = [
    sys.executable, "train.py",
    "--devices", str(DEVICE_ID),
    "--config", str(GENERATED_CONFIG),
]
run(train_command, cwd=PROJECT_DIR)
assert checkpoint.is_file(), f"checkpoint가 생성되지 않았습니다: {checkpoint}"
assert (WEIGHTS_DIR / "mlflow-run-id.txt").is_file()
print("fresh training 완료:", checkpoint)

## 6. MLflow history와 metric 전달 검증

MLflow run 화면의 요약값은 각 metric의 최신값이지만, `get_metric_history()`는 모든 step을 반환합니다. 아래 셀은 다음을 검사합니다.

- train metric이 여러 step에 남아 있는가
- 동일 metric의 step이 중복되거나 덮어써지지 않았는가
- DINO raw/scaled loss 또는 Cascade component/total loss가 올바른 key로 전달됐는가
- 평가 AP가 `eval/` namespace에 기록됐는가

In [ ]:
import math
import pandas as pd
from mlflow.tracking import MlflowClient

TRACKING_URI = f"sqlite:///{MLFLOW_DB}"
EXPERIMENT_NAME = f"colab-{BACKEND}-smoke"
client = MlflowClient(tracking_uri=TRACKING_URI)
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
assert experiment is not None, "MLflow experiment가 없습니다."
RUN_ID = (WEIGHTS_DIR / "mlflow-run-id.txt").read_text(encoding="utf-8").strip()
run_info = client.get_run(RUN_ID)
assert run_info.info.experiment_id == experiment.experiment_id
metric_keys = sorted(run_info.data.metrics)
print("run_id:", RUN_ID)
print("tags:", {key: run_info.data.tags.get(key) for key in ("detector", "action", "distributed")})
print("metric keys:")
print(*metric_keys, sep=chr(10))

def history(key, run_id=RUN_ID):
    return sorted(
        client.get_metric_history(run_id, key),
        key=lambda item: (item.step, item.timestamp),
    )

def values_by_step(key):
    points = history(key)
    steps = [item.step for item in points]
    assert len(steps) == len(set(steps)), f"{key}에 중복 step이 있습니다."
    return {item.step: item.value for item in points}

PRIMARY_METRIC = "train/loss" if BACKEND == "dino" else "train/total_loss"
primary_history = history(PRIMARY_METRIC)
assert len(primary_history) > 1, f"{PRIMARY_METRIC} history가 충분하지 않습니다."
primary_steps = [item.step for item in primary_history]
assert len(primary_steps) == len(set(primary_steps)), "동일 metric/step이 중복 기록됐습니다."

if BACKEND == "dino":
    assert "train/loss_bbox" in metric_keys, "raw bbox loss가 없습니다."
    assert "train/loss_bbox_scaled" in metric_keys, "scaled bbox loss가 없습니다."
    assert "train_epoch/loss" in metric_keys, "epoch 평균 loss가 없습니다."
    assert "eval/bbox_mAP" in metric_keys, "DINO evaluation mAP가 없습니다."
    component_keys = [
        key for key in metric_keys
        if key.startswith("train/loss_") and key.endswith("_scaled")
    ]
else:
    assert "eval/bbox/AP" in metric_keys, "Cascade evaluation AP가 eval namespace에 없습니다."
    assert not any(key.startswith("train/bbox/") for key in metric_keys)
    component_keys = [
        key for key in metric_keys
        if key.startswith("train/loss_") and key != "train/total_loss"
    ]

component_histories = {key: values_by_step(key) for key in component_keys}
checked_steps = 0
for item in primary_history:
    if all(item.step in values for values in component_histories.values()):
        component_sum = sum(values[item.step] for values in component_histories.values())
        assert math.isclose(item.value, component_sum, rel_tol=1e-5, abs_tol=1e-6), (
            item.step, item.value, component_sum
        )
        checked_steps += 1
assert checked_steps > 0, "total/component loss를 같은 step에서 비교하지 못했습니다."
history_before_resume = len(primary_history)
print(f"검증 완료: {PRIMARY_METRIC} points={history_before_resume}, loss_sum_steps={checked_steps}")

In [ ]:
import matplotlib.pyplot as plt

plot_keys = [PRIMARY_METRIC]
plot_keys += [key for key in metric_keys if key.startswith("eval/")][:3]
fig, axes = plt.subplots(len(plot_keys), 1, figsize=(10, 3 * len(plot_keys)), squeeze=False)
for axis, key in zip(axes[:, 0], plot_keys):
    points = history(key)
    axis.plot([point.step for point in points], [point.value for point in points], marker="o")
    axis.set(title=key, xlabel="MLflow step", ylabel="value")
    axis.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Resume 및 동일 MLflow run 검증

epoch 수만 2로 늘리고 `--resume`을 전달합니다. `recent.pth`에서 model/optimizer/scheduler/scaler를 복원하고, `mlflow-run-id.txt`에 저장된 기존 run ID를 다시 사용해야 합니다.

In [ ]:
if RUN_RESUME_TEST:
    resumed_settings = yaml.safe_load(GENERATED_CONFIG.read_text(encoding="utf-8"))
    resumed_settings["training"]["epochs"] = RESUME_EPOCHS
    GENERATED_CONFIG.write_text(
        yaml.safe_dump(resumed_settings, sort_keys=False, allow_unicode=True),
        encoding="utf-8",
    )
    run(train_command + ["--resume"], cwd=PROJECT_DIR)

    client = MlflowClient(tracking_uri=TRACKING_URI)
    active_run_id = (WEIGHTS_DIR / "mlflow-run-id.txt").read_text(encoding="utf-8").strip()
    resumed_run = client.get_run(active_run_id)
    resumed_history = history(PRIMARY_METRIC, run_id=active_run_id)
    assert resumed_run.data.tags.get("resumed") == "true"
    resumed_steps = [item.step for item in resumed_history]
    assert len(resumed_steps) == len(set(resumed_steps))
    if active_run_id == RUN_ID:
        assert len(resumed_history) > history_before_resume
        resume_artifacts = client.list_artifacts(active_run_id, "resume-configs/resume-1")
        assert {Path(item.path).name for item in resume_artifacts} == {"effective-config.yaml", "runtime.yaml"}
        resume_mode = "same run"
    else:
        assert resumed_run.data.tags.get("resume_of")
        assert resumed_history
        resume_mode = "linked retry attempt"
    print(
        f"resume 검증 완료: mode={resume_mode}, run_id={active_run_id}, "
        f"history_points={len(resumed_history)}"
    )
else:
    print("RUN_RESUME_TEST=False: resume 검증을 건너뜁니다.")

## 8. 저장소 MLflow 계약 테스트

GPU 학습 결과 검증과 별도로, 저장소의 테스트가 DINO epoch history, Cascade namespace/current-step filtering, raw/scaled loss naming, resume run 연결을 검사합니다.

In [ ]:
run([
    sys.executable, "-m", "pytest", "-q",
    "tests/test_contracts.py",
    "-k", "mlflow or epoch_losses or resumed_training",
], cwd=PROJECT_DIR)

run([sys.executable, "-m", "compileall", "-q", "train.py", "inference.py", "src", "scripts", "tests"], cwd=PROJECT_DIR)

if RUN_FULL_UNIT_TESTS:
    run([sys.executable, "-m", "pytest", "-q"], cwd=PROJECT_DIR)
else:
    print("RUN_FULL_UNIT_TESTS=False: 전체 unit test는 건너뜁니다.")

## 9. MLflow UI 안내

Colab의 비공개 port proxy API에는 의존하지 않습니다. 위의 `MlflowClient` 표와 그래프가 SQLite 원본을 직접 조회합니다. 로컬로 DB를 내려받은 뒤 아래 명령으로 UI를 열 수 있습니다.

In [ ]:
print(f"mlflow ui --backend-store-uri sqlite:///{MLFLOW_DB}")
print("Colab에서는 위 history/plot 셀을 기준 검증 경로로 사용합니다.")

## 산출물

- subset: `/content/dit-layout-colab/publaynet-subset-t<train>-v<val>-s<seed>`
- effective Colab YAML: `/content/dit-layout-colab/<backend>-colab.yaml`
- checkpoint: `/content/dit-layout-colab/output-<backend>/weights/recent.pth`
- MLflow DB: `/content/dit-layout-colab/mlflow.db`
- MLflow run 연결 파일: `weights/mlflow-run-id.txt`

Colab VM이 종료되면 `/content`가 삭제됩니다. 필요한 checkpoint와 MLflow DB는 종료 전에 Google Drive로 복사하세요.